# SmolDocling Tutorial for Document Processing & RAG Chunking

Vision-Language Models (VLM) recently became a big thing. Let's take a look how we can use them in document parsing.

This notebook demonstrates:

1. Installing and loading SmolDocling  
2. Converting a PDF into a DoclingDocument  
3. Extracting structure (e.g. table of contents)  
4. Chunking the document into smaller pieces suitable for Retrieval-Augmented Generation (RAG)  
5. Saving the chunks to `data/preprocessed_SmolDocling/`

## Imports and Device Configuration

In [ ]:
from pprint import pprint
import os
import torch
from docling_core.types.doc import DoclingDocument
from docling_core.types.doc.document import DocTagsDocument
from docling.document_converter import DocumentConverter

from transformers import AutoTokenizer
import os

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")


## Load and Convert Your Document

In [ ]:
# Path to your PDF (can also be a URL)
source = "/Users/philipp.schwartenbeck/Desktop/rag-llm-applications/data/raw/Geschäftsordnung  Trust Council.pdf"

# Initialize the converter and run
converter = DocumentConverter()
result = converter.convert(source)

# The converted document
doc: DoclingDocument = result.document

#### Task 1.1 Inspect doc and try it with different documents and document types! What do you see?

## Inspecting the Document

### Export as Markdown

In [ ]:
md = doc.export_to_markdown()
print(md[:1200])  # print the first 500 characters

### Export as Dict (Structured)

In [ ]:
doc_dict = doc.export_to_dict()

pprint(doc_dict)

In [ ]:
print("doc_dict keys:", doc_dict.keys())

### Task 1.2 Inspect the Content of this dict!

In [ ]:
# show the first 10 text objects
import json
for i, txt in enumerate(doc_dict["texts"][:10]):
    print(f"--- text #{i} ---")
    print(json.dumps(txt, indent=2))


## Extracting the Table of Contents

In [ ]:
# Get individual text blocks
raw_texts = doc_dict["texts"]

raw_texts

Check what the texts contain:

In [ ]:
for t in raw_texts:
    print(t["text"])
    print("===")

Check labels of raw texts

In [ ]:
for t in raw_texts:
    print(t.get("label"))

Now, print the table of contents

In [ ]:
header_idxs = [i for i, t in enumerate(raw_texts) if t.get("label") == "section_header"]

for chapter, idx in enumerate(header_idxs):
    print(f"{chapter+1}. {raw_texts[idx]['text']}")

## Chunking the Document for RAG

Build sections based on headers - What is going on here?

In [ ]:
sections = []
for idx, start in enumerate(header_idxs):
    # determine end of this section
    end = header_idxs[idx + 1] if idx + 1 < len(header_idxs) else len(raw_texts)
    
    # title is the header text 
    title = raw_texts[start]["text"].strip()
    
    # collect all body blocks (text + list_item) under this header
    body_parts = []
    for t in raw_texts[start + 1 : end]:
        lbl = t.get("label")
        if lbl in ("text", "list_item"):
            # optionally, for list_item you can add a bullet:
            if lbl == "list_item":
                body_parts.append(f"- {t['text'].strip()}")
            else:
                body_parts.append(t["text"].strip())
    
    full_text = "\n\n".join(body_parts)
    sections.append({"title": title, "text": full_text})

print(f"Built {len(sections)} sections via headers.")

Adjust max_tokens to taste

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("ds4sd/SmolDocling-256M-preview")

In [ ]:
out_dir = "data/preprocessed_SmolDocling"
os.makedirs(out_dir, exist_ok=True)

In [ ]:

chunked_files = []

for sec_idx, sec in enumerate(sections):
    tokens = tokenizer.tokenize(sec["text"])
    print(f"Section {sec_idx+1} has {len(tokens)} tokens.")
    chunk_text   = tokenizer.convert_tokens_to_string(tokens)
    
    fname        = f"section{sec_idx+1}.txt"
    path         = os.path.join(out_dir, fname)
    
    with open(path, "w", encoding="utf-8") as f:
        f.write(chunk_text)
    chunked_files.append(path)

print(f"Saved {len(chunked_files)} chunks under `{out_dir}`")

#### Task 1.3 In the above loop, implement a way of controlling a specific chunk size!

In [ ]:
max_tokens = 500
chunked_files = []

...

print(f"Saved {len(chunked_files)} chunks under `{out_dir}`")

Now save as a json and txt:

In [ ]:
chunk_out = "data/preprocessed_SmolDocling/chunks"
os.makedirs(chunk_out, exist_ok=True)

for path in chunked_files:
    # read the text again
    with open(path, encoding="utf-8") as f:
        chunk_text = f.read()
    # prepare metadata
    meta = {
        "chunk_filename": os.path.basename(path),
        "text": chunk_text
    }
    base = os.path.splitext(os.path.basename(path))[0]
    # 1) JSON
    with open(os.path.join(chunk_out, f"{base}.json"), "w", encoding="utf-8") as jf:
        json.dump(meta, jf, ensure_ascii=False, indent=2)
    # 2) TXT (just the raw chunk)
    with open(os.path.join(chunk_out, f"{base}.txt"), "w", encoding="utf-8") as tf:
        tf.write(chunk_text)

print(f"Saved {len(chunked_files)} chunks as JSON+TXT in `{chunk_out}`")

### Verifying Your Chunks

In [ ]:
# List first few chunk files and display their content
for p in chunked_files[:5]:
    print(f"\n--- {p} ---")
    print(open(p, encoding="utf-8").read()[:300], "...\n")